# Offline Reinforcement Learning Tutorial

This notebook demonstrates how to use offline RL algorithms (CQL and IQL) to learn from fixed datasets without environment interaction.

## What is Offline RL?

Offline RL learns from pre-collected datasets without interacting with the environment during training. This is useful when:
- Environment interaction is expensive or dangerous
- You have historical logged data
- You want to improve policies from suboptimal demonstrations

## Algorithms Covered

1. **CQL (Conservative Q-Learning)**: Uses conservative Q-value estimates to avoid overestimation
2. **IQL (Implicit Q-Learning)**: Uses expectile regression for stable offline learning

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from rl_llm_toolkit import RLEnvironment
from rl_llm_toolkit.agents.cql import CQLAgent
from rl_llm_toolkit.agents.iql import IQLAgent

print("✅ Imports successful!")

## Step 1: Create Environment and Collect Dataset

First, we'll collect an offline dataset using a random policy.

In [ ]:
env = RLEnvironment("CartPole-v1")

print(f"Environment: {env.env_name}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

In [ ]:
# Create CQL agent for dataset collection
collector = CQLAgent(env=env, seed=42)

# Collect dataset with random policy
print("Collecting offline dataset...")
dataset = collector.collect_dataset(
    num_episodes=100,
    policy="random",
)

print(f"\n✅ Dataset collected: {len(dataset)} transitions")

# Analyze dataset
rewards = [t['reward'] for t in dataset]
print(f"\nDataset Statistics:")
print(f"  Total reward: {sum(rewards):.2f}")
print(f"  Mean reward: {np.mean(rewards):.2f}")
print(f"  Std reward: {np.std(rewards):.2f}")

## Step 2: Train CQL Agent

Conservative Q-Learning adds a penalty to overestimated Q-values.

In [ ]:
# Create and train CQL agent
cql_agent = CQLAgent(
    env=env,
    learning_rate=3e-4,
    batch_size=256,
    cql_weight=1.0,  # Conservative penalty weight
    seed=42,
)

cql_agent.load_dataset(dataset)

print("Training CQL agent...")
cql_results = cql_agent.train(
    total_timesteps=50000,
    log_interval=5000,
    eval_interval=10000,
    eval_episodes=10,
    progress_bar=True,
)

print(f"\n✅ CQL training complete!")
print(f"Total updates: {cql_results['updates']}")

## Step 3: Train IQL Agent

Implicit Q-Learning uses expectile regression for value learning.

In [ ]:
# Create and train IQL agent
iql_agent = IQLAgent(
    env=env,
    learning_rate=3e-4,
    batch_size=256,
    expectile=0.7,  # Expectile for value learning
    temperature=3.0,  # Temperature for policy extraction
    seed=42,
)

iql_agent.load_dataset(dataset)

print("Training IQL agent...")
iql_results = iql_agent.train(
    total_timesteps=50000,
    log_interval=5000,
    eval_interval=10000,
    eval_episodes=10,
    progress_bar=True,
)

print(f"\n✅ IQL training complete!")
print(f"Total updates: {iql_results['updates']}")

## Step 4: Evaluate and Compare

Let's evaluate both agents and compare with the dataset baseline.

In [ ]:
# Evaluate CQL
print("Evaluating CQL agent...")
cql_eval = cql_agent.evaluate(episodes=20, deterministic=True)

print(f"\nCQL Results:")
print(f"  Mean Reward: {cql_eval['mean_reward']:.2f} ± {cql_eval['std_reward']:.2f}")
print(f"  Min Reward: {cql_eval['min_reward']:.2f}")
print(f"  Max Reward: {cql_eval['max_reward']:.2f}")

# Evaluate IQL
print("\nEvaluating IQL agent...")
iql_eval = iql_agent.evaluate(episodes=20, deterministic=True)

print(f"\nIQL Results:")
print(f"  Mean Reward: {iql_eval['mean_reward']:.2f} ± {iql_eval['std_reward']:.2f}")
print(f"  Min Reward: {iql_eval['min_reward']:.2f}")
print(f"  Max Reward: {iql_eval['max_reward']:.2f}")

In [ ]:
# Comparison visualization
baseline_reward = np.mean(rewards)

algorithms = ['Random\n(Dataset)', 'CQL', 'IQL']
mean_rewards = [
    baseline_reward,
    cql_eval['mean_reward'],
    iql_eval['mean_reward']
]
std_rewards = [
    np.std(rewards),
    cql_eval['std_reward'],
    iql_eval['std_reward']
]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(algorithms))
bars = ax.bar(x, mean_rewards, yerr=std_rewards, capsize=10, 
              color=['gray', 'steelblue', 'coral'], alpha=0.8)

ax.set_xlabel('Algorithm', fontsize=12)
ax.set_ylabel('Mean Reward', fontsize=12)
ax.set_title('Offline RL Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(algorithms)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (bar, val) in enumerate(zip(bars, mean_rewards)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.1f}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Calculate improvements
cql_improvement = ((cql_eval['mean_reward'] - baseline_reward) / baseline_reward) * 100
iql_improvement = ((iql_eval['mean_reward'] - baseline_reward) / baseline_reward) * 100

print(f"\n{'='*60}")
print("Performance Improvement over Random Policy")
print(f"{'='*60}")
print(f"CQL: +{cql_improvement:.1f}%")
print(f"IQL: +{iql_improvement:.1f}%")

## Step 5: Visualize Training Progress

In [ ]:
# Get training statistics
cql_stats = cql_agent.get_training_stats()
iql_stats = iql_agent.get_training_stats()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot losses
if cql_stats['stats']['losses']:
    ax1.plot(cql_stats['stats']['losses'], label='CQL', alpha=0.7)
if iql_stats['stats']['losses']:
    ax1.plot(iql_stats['stats']['losses'], label='IQL', alpha=0.7)

ax1.set_xlabel('Update Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot episode rewards (if available)
ax2.axhline(y=baseline_reward, color='gray', linestyle='--', 
            label='Random Policy', linewidth=2)
ax2.axhline(y=cql_eval['mean_reward'], color='steelblue', 
            linestyle='--', label='CQL Final', linewidth=2)
ax2.axhline(y=iql_eval['mean_reward'], color='coral', 
            linestyle='--', label='IQL Final', linewidth=2)

ax2.set_xlabel('Episode')
ax2.set_ylabel('Reward')
ax2.set_title('Performance Comparison')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 6: Save Models

In [ ]:
# Save trained models
model_dir = Path("./models/offline_rl")
model_dir.mkdir(parents=True, exist_ok=True)

cql_path = model_dir / "cql_cartpole.pt"
iql_path = model_dir / "iql_cartpole.pt"

cql_agent.save(cql_path)
iql_agent.save(iql_path)

print(f"✅ Models saved:")
print(f"  CQL: {cql_path}")
print(f"  IQL: {iql_path}")

## Step 7: Test Loaded Models

In [ ]:
# Load and test CQL model
loaded_cql = CQLAgent(env=env)
loaded_cql.load(cql_path)

test_results = loaded_cql.evaluate(episodes=5, deterministic=True)
print(f"Loaded CQL performance: {test_results['mean_reward']:.2f}")

# Verify it matches
assert abs(test_results['mean_reward'] - cql_eval['mean_reward']) < 10, "Model loading issue!"
print("✅ Model loading verified!")

## Key Takeaways

1. **Offline RL** enables learning from fixed datasets without environment interaction
2. **CQL** uses conservative Q-value estimates to prevent overestimation
3. **IQL** uses expectile regression for stable offline learning
4. Both algorithms significantly improve over the random baseline
5. Offline RL is useful for:
   - Learning from logged data
   - Safe policy improvement
   - Expensive or dangerous environments

## Next Steps

- Try different dataset qualities (expert vs random)
- Experiment with hyperparameters (CQL weight, expectile)
- Apply to more complex environments
- Combine with online fine-tuning

In [ ]:
env.close()
print("\n✅ Tutorial complete!")